In [ ]:
%matplotlib inline

from itertools import chain
import json
import os
from typing import Any, Hashable, Tuple

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    precision_recall_fscore_support,
)
from tqdm import tqdm

from kibad_llm.config import PROJ_ROOT
from kibad_llm.dataset.json import read_and_preprocess
from kibad_llm.dataset.utils import merge_references_into_predictions
from kibad_llm.utils.dictionary import flatten_dict_simple

# swith to project root to use same paths as in commands
os.chdir(PROJ_ROOT)
# set wider column width for displaying pandas data frames
pd.set_option("max_colwidth", 400)

In [ ]:
predictions_file = "data/prediction_results/2025-11-25_19-49-41/predictions.jsonl"
ground_truth_file = "data/interim/faktencheck-db/faktencheck-db-converted_2025-11-05.jsonl"
FLATTEN_DICTS = True

In [ ]:
# this below is the exact same code used in the evaluation workflow


def load_predictions_with_references(
    pred_file: str, ref_file: str
) -> dict[Hashable, dict[str, dict]]:
    predictions = read_and_preprocess(
        file=pred_file,
        id_key="file_name",
        process_id=lambda x: os.path.splitext(x)[0],
        preprocess=lambda x: x.get("structured", None),
    )
    references = read_and_preprocess(
        file=ref_file,
        id_key="zotitem_ptr_id",
    )
    result = merge_references_into_predictions(predictions, references)
    return result


dataset = load_predictions_with_references(predictions_file, ground_truth_file)
f"loaded {len(dataset)} predictions"

In [ ]:
if FLATTEN_DICTS:

    # flatten_dict_simple does not handle nested entries, so we need to call it for prediction and reference individually.
    # Also, prediction may be None for some documents, so we use "{}" in these cases.
    dataset = {
        k: {
            "prediction": flatten_dict_simple(v["prediction"] or {}),
            "reference": flatten_dict_simple(v["reference"]),
        }
        for k, v in dataset.items()
    }

In [ ]:
predictions_df = pd.DataFrame.from_dict({k: v["prediction"] for k, v in dataset.items()}).T
reference_df = pd.DataFrame.from_dict({k: v["reference"] for k, v in dataset.items()}).T

print(f"available columns:\n{sorted(set(reference_df.columns) & set(predictions_df.columns))}")

In [ ]:
def get_column(col: str, replace_nan: Any | None = None) -> pd.DataFrame:
    result = pd.concat({"prediction": predictions_df[col], "reference": reference_df[col]}, axis=1)
    if replace_nan is not None:
        # we can not use .fillna(), since that does not work with lists
        mask = result.isna()
        result[mask] = result[mask].map(lambda _: replace_nan)

    return result


get_column("taxa.german_name", [])

In [ ]:
na_label = "_NA_"


def align_row(pred_list, true_list, missing_label=na_label):
    """
    pred_list and true_list are lists of labels (str), possibly empty
    Returns (union_sorted, aligned_pred, aligned_true) tuples:
      - union_sorted: alphabetically sorted union of labels
      - aligned_pred: list of same length as union_sorted, label if in pred_list else 'missing_label'
      - aligned_true: list of same length as union_sorted, label if in true_list else 'missing_label'
    """
    union = sorted(set(pred_list) | set(true_list))
    aligned_pred = [label if label in pred_list else missing_label for label in union]
    aligned_true = [label if label in true_list else missing_label for label in union]

    return union, aligned_pred, aligned_true


column_name = "taxa.german_name"
result_df = get_column(column_name, [])
# Zeilenweise anwenden und neue Spalten anlegen
aligned = result_df.apply(lambda row: align_row(row["prediction"], row["reference"]), axis=1)
# res ist eine Series mit Tupeln (union, aligned_pred, aligned_true) — wir entpacken:
result_df[["union_labels", "aligned_pred", "aligned_true"]] = pd.DataFrame(
    aligned.tolist(), index=aligned.index
)
result_df.head()

In [ ]:
# have a look only at rows which contain either a reference or a prediction
filter_empty_union_df = result_df[~result_df.union_labels.isin([[]])]
filter_empty_union_df[1:20]

In [ ]:
# 2E9XWUUE -> Referenzfehler: reference sind eher Artengruppen(?), nicht Spezies. Keine Predictions
# 2EUNPHDZ -> Predictionfehler: nichts predicted
# 2FDHAHAR -> lateinische Ein-Wort-Namen statt deutscher Namen predicted, aber reference ist auch eine Mischung aus deutschen und lateinischen Begriffen
# 3LGPK6BL -> Referenzfehler: keine Spezies in Referenz, eher allgemeiner Begriff?
# 3TWLQG7F -> Predictionfehler: nichts predicted
# 3Z5BFIBL -> Referenzfehler: reference sind teilweise Artengruppen(?)
# 49Z7RPDP -> Referenzfehler: reference sind eher Artengruppen(?)
# 4FCS62W5 -> Predictionfehler: nichts predicted
# 4JF8F3WL -> Predictionfehler: nichts predicted; Referenzfehler: reference sind eher Artengruppen(?)
# 4Z67G9T5 -> Predictionfehler: nichts predicted; Referenzfehler: reference sind eher Artengruppen(?)
# 57CBE476 -> Referenzfehler: reference sind eher Artengruppen(?)
# 5CZNRP7X -> Predictionfehler: "Atlantischer" fehlt bei 3 Arten
# 5LC5W7KF -> Referenzfehler: reference sind eher Artengruppen(?)
# 5SIYLM9W -> Referenzfehler: reference sind eher Artengruppen(?)

In [ ]:
# flatten list of lists
union_labels_flat = sorted(list(set(list(chain.from_iterable(result_df["union_labels"])))))
y_pred_flat = list(chain.from_iterable(result_df["aligned_pred"]))
y_true_flat = list(chain.from_iterable(result_df["aligned_true"]))

# F1-Score berechnen (für Multiclass)
score = precision_recall_fscore_support(
    y_true_flat, y_pred_flat, average="micro", labels=union_labels_flat
)
print(
    f"{column_name}: p={score[0]:3.3}, r={score[1]:3.3}, f1={score[2]:3.3}, support={len([y for y in y_true_flat if y != na_label])}"
)

In [ ]:
# also compute the confusion matrix and plot it. not really useful atm
# cm = confusion_matrix(y_true_flat, y_pred_flat, labels=union_labels_flat + na_label])
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=union_labels_flat + [na_label])
# disp.plot()
# plt.show()

In [ ]:
# biodiversity_variable -> gold label sind oft Englisch, z.B. 'Abundance of mice', 'Abundance of voles'
# transformation_potential -> nie predicted
# in location sind gold werte wie '(('country', 'Australia'), ('federal_state', 'Baden-Württemberg'), ('name', 'Western Australian coast'))' ???
# in location sind gold werte wie (('country', 'Belgium'),) |   (('country', 'Cambodia'), ('name', 'Cambodia'))
# species mismatches wegen kleiner unterschiede, z.B predicted = ('german_name', 'Acker-Hasenohr, Rundblättriges Hasenohr'), ('scientific_name', 'Bupleurum rotundifolium L.'), ('species_group', 'Gefäßpflanzen')
# gold = (('german_name', 'Acker-Hasenohr, Rundblättriges Hasenohr'), ('scientific_name', 'Bupleurum rotundifolium'), ('species_group', 'Gefäßpflanzen')) -> einziger Unterschied ist ' L.' im scientific name
# ansonsten sehen predictions für taxa eigentlich gut aus (immer deutscher oder lateinischer Begriff)